## Contingency table

#### Packages,Domains and EDAM categories

In [24]:
import os
import pandas as pd
import numpy as np


In [25]:

# Domains and EDAM categories
domains = [
    "Metagenomic",
    "Neuroimaging",
    "Phylogeny",
    "Single_Cell",
    "Systems_biology",
    "Genetic_variant",
    "Microscopy"
]
edam_categories = ["data", "formats", "operations", "topics"]

In [26]:
# Function to clean column names
def clean_columns(df):
    df.columns = (
        df.columns
        .str.strip()              
        .str.replace(r"\s+", " ", regex=True)  
    )
    return df

#### Rows kept through all domains and files

In [27]:
processed_dfs = []

# Loop through files
for domain in domains:
    for category in edam_categories:
        file_path = f"{domain}/{domain}_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_{category}.tsv"
        
        if os.path.exists(file_path):
            df = pd.read_csv(file_path, sep="\t")
            df = clean_columns(df)
            
            # Add metadata
            df["domain"] = domain
            df["edam_category"] = category
            
            processed_dfs.append(df)
            print(f"Processed {file_path}: {len(df)} rows kept.")
        else:
            print(f"Warning: File not found - {file_path}")


Processed Metagenomic/Metagenomic_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_data.tsv: 24 rows kept.
Processed Metagenomic/Metagenomic_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_formats.tsv: 68 rows kept.
Processed Metagenomic/Metagenomic_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_operations.tsv: 71 rows kept.
Processed Metagenomic/Metagenomic_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_topics.tsv: 70 rows kept.
Processed Neuroimaging/Neuroimaging_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_data.tsv: 17 rows kept.
Processed Neuroimaging/Neuroimaging_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_formats.tsv: 68 rows kept.
Processed Neuroimaging/Neuroimaging_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_operations.tsv: 71 rows kept.
Processed Neuroimaging/Neuroimaging_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_topics.tsv: 10 rows kept.
Processed Phylogeny/Phylogeny_gold_standard_consensus_expert_LLM_mapping_t

#### Combine everything into one dataframe

In [ ]:
# Combine into master DataFrame
if processed_dfs:
    master_df = pd.concat(processed_dfs, ignore_index=True)
    display(master_df.head())
    print("\nColumns:")
    print(master_df.columns.tolist())
else:
    print("No data processed.")


Total validated rows in Master DataFrame: 1118


,tools,annotation,candidate EDAM term label,Validation expert,Commentaire expert 1,Commentaire expert 2,hasExactmatch,hasBroader,hasNarrower,EDAM term URI,Comment,domain,edam_category,Commentaire expert 3
0,MetaBAT2,abundance profiling,Biodiversity data,Yes,Oui mais le terme Biodiversity data est bien p...,NaN,No,Yes,No,http://edamontology.org/data_3707,NaN,Metagenomic,data,NaN
1,"Kraken2, mOTUs",abundance table,Biodiversity data,Yes,NaN,NaN,No,Yes,No,http://edamontology.org/data_3707,NaN,Metagenomic,data,NaN
2,Anvi’o,annotated contigs,Sequence Assembly,No,NaN,NaN,No,Yes,No,http://edamontology.org/data_0925,NaN,Metagenomic,data,NaN
3,"Anvi’o, CONCOCT, MetaBAT2",bins,Sequence set,No,NaN,NaN,No,Yes,No,http://edamontology.org/data_0850,NaN,Metagenomic,data,NaN
4,MetaBAT2,co-abundance,Biodiversity data,No,NaN,NaN,No,Yes,No,http://edamontology.org/data_3707,NaN,Metagenomic,data,NaN



Columns:
['tools', 'annotation', 'candidate EDAM term label', 'Validation expert', 'Commentaire expert 1', 'Commentaire expert 2', 'hasExactmatch', 'hasBroader', 'hasNarrower', 'EDAM term URI', 'Comment', 'domain', 'edam_category', 'Commentaire expert 3']


In [8]:
master_df.to_csv(
    "All_Domains_ground_truth_consensus_expert_LLM_mapping_terms_EDAM.tsv",
    sep="\t",
    index=False
)

print("Saved as All_Domains_ground_truth_consensus_expert_LLM_mapping_terms_EDAM.tsv")

Saved as All_Domains_ground_truth_consensus_expert_LLM_mapping_terms_EDAM.tsv


In [ ]:
# Concatenate
master_df = pd.concat(processed_dfs, ignore_index=True)

master_df = master_df.drop_duplicates(subset=[
    "domain", "edam_category", "candidate EDAM term label", 
    "Validation expert", "hasExactmatch", "hasBroader", "hasNarrower"
])

In [32]:
no_candidate = (
    (master_df["candidate EDAM term label"] == "no_matching_term") &
    (master_df["hasExactmatch"] == "No") &
    (master_df["hasBroader"] == "No") &
    (master_df["hasNarrower"] == "No")
)

unvalidated = (
    master_df["candidate EDAM term label"].notna() &
    (master_df["candidate EDAM term label"] != "no_matching_term") &
    (master_df["Validation expert"] == "No")
)

validated_exact = (
    master_df["candidate EDAM term label"].notna() &
    (master_df["candidate EDAM term label"] != "no_matching_term") &
    (master_df["Validation expert"] == "Yes") &
    (master_df["hasExactmatch"] == "Yes")
)

validated_broader = (
    master_df["candidate EDAM term label"].notna() &
    (master_df["candidate EDAM term label"] != "no_matching_term") &
    (master_df["Validation expert"] == "Yes") &
    (master_df["hasBroader"] == "Yes")
)

validated_narrower = (
    master_df["candidate EDAM term label"].notna() &
    (master_df["candidate EDAM term label"] != "no_matching_term") &
    (master_df["Validation expert"] == "Yes") &
    (master_df["hasNarrower"] == "Yes")
)

In [33]:
contingency_table = pd.DataFrame({
    "Category": [
        "Annotation without candidate EDAM term",
        "Annotation with unvalidated EDAM class",
        "Annotation validated (Exact match)",
        "Annotation validated (Broader)",
        "Annotation validated (Narrower)"
    ],
    "Count": [
        no_candidate.sum(),
        unvalidated.sum(),
        validated_exact.sum(),
        validated_broader.sum(),
        validated_narrower.sum()
    ]
})

display(contingency_table)

,Category,Count
0,Annotation without candidate EDAM term,11
1,Annotation with unvalidated EDAM class,136
2,Annotation validated (Exact match),199
3,Annotation validated (Broader),202
4,Annotation validated (Narrower),15


In [ ]:
contingency_table = pd.DataFrame({
    "Category": [
        "Annotation without candidate EDAM term",
        "Annotation with unvalidated EDAM class",
        "Annotation validated (Exact match)",
        "Annotation validated (Broader)",
        "Annotation validated (Narrower)"
    ],
    "Count": [
        no_candidate.sum(),
        unvalidated.sum(),
        validated_exact.sum(),
        validated_broader.sum(),
        validated_narrower.sum()
    ]
})

display(contingency_table)

,Category,Count
0,Annotation without candidate EDAM term,30
1,Annotation with unvalidated EDAM class,133
2,Annotation validated (Exact match),153
3,Annotation validated (Broader),270
4,Annotation validated (Narrower),15


In [12]:
def compute_counts(df):
    return pd.Series({
        "No candidate": (
            (df["candidate EDAM term label"] == "no_matching_term") & 
            (df["hasExactmatch"] == "No") & 
            (df["hasBroader"] == "No") & 
            (df["hasNarrower"] == "No")
        ).sum(),
        "Unvalidated": (
            (df["candidate EDAM term label"].notna()) & 
            (df["candidate EDAM term label"] != "no_matching_term") & 
            (df["Validation expert"] == "No")
        ).sum(),
        "Exact": (
            (df["Validation expert"] == "Yes") & 
            (df["hasExactmatch"] == "Yes") & 
            (df["candidate EDAM term label"] != "no_matching_term")
        ).sum(),
        "Broader": (
            (df["Validation expert"] == "Yes") & 
            (df["hasBroader"] == "Yes") & 
            (df["candidate EDAM term label"] != "no_matching_term")
        ).sum(),
        "Narrower": (
            (df["Validation expert"] == "Yes") & 
            (df["hasNarrower"] == "Yes") & 
            (df["candidate EDAM term label"] != "no_matching_term")
        ).sum(),
    })

domain_table = master_df.groupby("domain").apply(compute_counts)

display(domain_table)

/tmp/ipykernel_205537/2272126979.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  domain_table = master_df.groupby("domain").apply(compute_counts)


,No candidate,Unvalidated,Exact,Broader,Narrower
domain,,,,,
Genetic_variant,0,3,25,33,6
Metagenomic,0,15,31,16,3
Microscopy,4,44,60,49,1
Neuroimaging,3,4,10,17,0
Phylogeny,0,2,10,13,2
Single_Cell,1,5,16,12,2
Systems_biology,3,63,47,62,1


In [12]:
def compute_counts(df):
    return pd.Series({
        "No candidate": (
            (df["candidate EDAM term label"] == "no_matching_term") & 
            (df["hasExactmatch"] == "No") & 
            (df["hasBroader"] == "No") & 
            (df["hasNarrower"] == "No")
        ).sum(),
        "Unvalidated": (
            (df["candidate EDAM term label"].notna()) & 
            (df["candidate EDAM term label"] != "no_matching_term") & 
            (df["Validation expert"] == "No")
        ).sum(),
        "Exact": (
            (df["Validation expert"] == "Yes") & 
            (df["hasExactmatch"] == "Yes") & 
            (df["candidate EDAM term label"] != "no_matching_term")
        ).sum(),
        "Broader": (
            (df["Validation expert"] == "Yes") & 
            (df["hasBroader"] == "Yes") & 
            (df["candidate EDAM term label"] != "no_matching_term")
        ).sum(),
        "Narrower": (
            (df["Validation expert"] == "Yes") & 
            (df["hasNarrower"] == "Yes") & 
            (df["candidate EDAM term label"] != "no_matching_term")
        ).sum(),
    })

domain_table = master_df.groupby("domain").apply(compute_counts)

display(domain_table)

/tmp/ipykernel_174521/2272126979.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  domain_table = master_df.groupby("domain").apply(compute_counts)


,No candidate,Unvalidated,Exact,Broader,Narrower
domain,,,,,
Genetic_variant,0,3,25,45,7
Metagenomic,0,19,31,20,3
Microscopy,0,10,10,30,0
Neuroimaging,5,8,10,25,0
Phylogeny,0,2,10,17,2
Single_Cell,3,6,18,20,2
Systems_biology,22,85,49,113,1


#### Keep URI EDAM term validated

In [13]:
validated_df = master_df[
    (master_df["candidate EDAM term label"].notna()) &
    (master_df["candidate EDAM term label"] != "no_matching_term") &
    (master_df["Validation expert"] == "Yes")
].copy()

validated_df = validated_df[
    [
        "EDAM term URI",
        "candidate EDAM term label",
        "tools",
        "annotation",
        "domain",
        "edam_category"
    ]
]

In [13]:
validated_df

,EDAM term URI,candidate EDAM term label,tools,annotation,domain,edam_category
0,http://edamontology.org/data_3707,Biodiversity data,MetaBAT2,abundance profiling,Metagenomic,data
1,http://edamontology.org/data_3707,Biodiversity data,"Kraken2, mOTUs",abundance table,Metagenomic,data
5,http://edamontology.org/data_0925,Sequence Assembly,"Anvi’o, CONCOCT, MEGAHIT, MetaBAT2, checkM2",contigs,Metagenomic,data
7,http://edamontology.org/operation_3672,Gene functional annotation,Anvi’o,functional annotations,Metagenomic,data
9,http://edamontology.org/data_2968,Image,Anvi’o,image,Metagenomic,data
...,...,...,...,...,...,...
950,http://edamontology.org/topic_3382,Imaging,EcClem,correlative microscopy,Microscopy,topics
951,http://edamontology.org/topic_3382,Imaging,Big-FISH,fluorescence in situ hybriditation microscopy,Microscopy,topics
954,http://edamontology.org/topic_3382,Imaging,CellPose,optical microscopy,Microscopy,topics
955,http://edamontology.org/topic_3308,Transcriptomics,Big-FISH,spatial transcriptomics,Microscopy,topics


In [14]:
tools_to_remove = ["htseq-count", "Bedtools_annotate", "deconvolutionLab2",
                   "Big-FISH", "AFNI", "MULTIXRANK"]

# Split tools column into list
tool_lists = validated_df["tools"].str.split(r",\s*")

# Keep rows where NOT all tools are in tools_to_remove
validated_df_filtered = validated_df[
    tool_lists.apply(lambda tools: not all(t in tools_to_remove for t in tools))
]
filtered_count = len(validated_df) - len(validated_df_filtered)
print(filtered_count)

40


In [15]:
validated_df_filtered

,EDAM term URI,candidate EDAM term label,tools,annotation,domain,edam_category
0,http://edamontology.org/data_3707,Biodiversity data,MetaBAT2,abundance profiling,Metagenomic,data
1,http://edamontology.org/data_3707,Biodiversity data,"Kraken2, mOTUs",abundance table,Metagenomic,data
5,http://edamontology.org/data_0925,Sequence Assembly,"Anvi’o, CONCOCT, MEGAHIT, MetaBAT2, checkM2",contigs,Metagenomic,data
7,http://edamontology.org/operation_3672,Gene functional annotation,Anvi’o,functional annotations,Metagenomic,data
9,http://edamontology.org/data_2968,Image,Anvi’o,image,Metagenomic,data
...,...,...,...,...,...,...
945,http://edamontology.org/operation_3443,Image analysis,CellPose,segmentation 2d,Microscopy,operations
946,http://edamontology.org/operation_3443,Image analysis,CellPose,segmentation 3d,Microscopy,operations
948,http://edamontology.org/topic_2229,Cell biology,CellPose,cell segmentation,Microscopy,topics
950,http://edamontology.org/topic_3382,Imaging,EcClem,correlative microscopy,Microscopy,topics


In [56]:
validated_df_filtered.to_csv(
    "EDAM_terms_URI_ground_truth_validated.tsv",
    sep="\t",
    index=False
)

print("Saved: EDAM_terms_URI_ground_truth_validated.tsv")

Saved: EDAM_terms_URI_ground_truth_validated.tsv
